# NLP — Carnet 06 : recherche d'information depuis zéro en C# (TF-IDF, BM25)

**Navigation** : [<< Précédent — 05. HMM et Viterbi](05-HMM-Viterbi.ipynb) | [Index de la série](README.md) | [Version Python](06-Recherche-Information-TFIDF-BM25.ipynb) | **06 (ce notebook)** | [Suivant — 07. Vecteurs de mots](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb)

La série a modélisé des mots, des séquences et des structures. Ce carnet inverse la question : au lieu de demander « quelle est la probabilité de ce texte ? », nous cherchons « quel document répond à cette requête ? ». Nous allons calculer et comparer, sans moteur externe, le comptage brut, TF-IDF cosinus et une variante pédagogique de BM25.

> **Public : Licence.** Prérequis : fréquences, logarithmes et vecteurs de mots. **Durée estimée : 45 minutes.** **Environnement : .NET 9 ou ultérieur, noyau .NET Interactive.**
>
> **Objectifs** : construire un corpus tokenisé et sa fréquence documentaire ; calculer TF-IDF et BM25 ; ordonner des documents avec des ex æquo déterministes ; mesurer Recall@k et analyser ce qu’il ne mesure pas.
>
> Les listes de termes sont déjà tokenisées : le carnet reproduit exactement le corpus miniature du jumeau Python. Nous conservons ses répétitions et ses accents tels quels, sans ajouter de pipeline de normalisation.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Security.Cryptography;
using System.Text;
using System.Text.Encodings.Web;
using System.Text.Json;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
record PlotSvg(string Markup);
Formatter.Register(typeof(PlotSvg),
    (value, writer) => ((System.IO.TextWriter)writer).Write(((PlotSvg)value).Markup),
    "text/html");
Console.WriteLine($"Kernel .NET Interactive prêt : runtime {Environment.Version}; cible pédagogique .NET 9, bibliothèques de base uniquement.");
Console.WriteLine("Formateur PlotSvg enregistré pour les figures SVG statiques.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Kernel .NET Interactive prêt : runtime 10.0.14; cible pédagogique .NET 9, bibliothèques de base uniquement.


Formateur PlotSvg enregistré pour les figures SVG statiques.


## 1. Un corpus miniature, avec une difficulté volontaire

Huit documents décrivent quatre thèmes. Le terme `note` est présent dans chacun : il constitue un bon test pour distinguer fréquence brute et information. Nous conservons l’ordre des termes et leurs répétitions, car ils définissent les fréquences TF et les longueurs utilisées plus loin.

### Exemple 1 — corpus embarqué

In [2]:
var corpus = new SortedDictionary<string, List<string>>(StringComparer.Ordinal)
{
    ["doc1"] = new() { "recherche", "documents", "pertinence", "classement", "note" },
    ["doc2"] = new() { "recherche", "recherche", "index", "inversion", "note" },
    ["doc3"] = new() { "recherche", "note" },
    ["doc4"] = new() { "compression", "entropie", "huffman", "code", "note" },
    ["doc5"] = new() { "compression", "compression", "entropie", "shannon", "note", "long" },
    ["doc6"] = new() { "jeux", "strategie", "equilibre", "note", "note", "note" },
    ["doc7"] = new() { "langue", "mots", "modeles", "probabilistes", "note" },
    ["doc8"] = new() { "langue", "mots", "mots", "modeles", "n-grammes", "probabilistes", "note", "long" }
};
int documentCount = corpus.Count;
var vocabulary = corpus.Values.SelectMany(tokens => tokens)
    .Distinct(StringComparer.Ordinal).OrderBy(term => term, StringComparer.Ordinal).ToArray();
var lengths = corpus.ToDictionary(pair => pair.Key, pair => pair.Value.Count, StringComparer.Ordinal);
var canonicalCorpus = JsonSerializer.Serialize(corpus);
string corpusSha256 = Convert.ToHexString(SHA256.HashData(Encoding.UTF8.GetBytes(canonicalCorpus))).ToLowerInvariant();
Console.WriteLine($"Corpus : {documentCount} documents, {vocabulary.Length} termes distincts; empreinte canonique SHA-256 {corpusSha256}.");
Console.WriteLine("Longueurs : " + string.Join(", ", lengths.Select(pair => $"{pair.Key}={pair.Value}")));
Console.WriteLine($"Le terme 'note' apparaît dans {corpus.Count(pair => pair.Value.Contains("note", StringComparer.Ordinal))}/{documentCount} documents.");

Corpus : 8 documents, 21 termes distincts; empreinte canonique SHA-256 9ef03be0c1ebab3edef40cfa7426fb83b1575d93d9e6e028d9cf1747c0979883.


Longueurs : doc1=5, doc2=5, doc3=2, doc4=5, doc5=6, doc6=6, doc7=5, doc8=8


Le terme 'note' apparaît dans 8/8 documents.


Le corpus contient des listes de jetons, pas des chaînes brutes à re-tokeniser. Ainsi, `doc2` contient deux occurrences de `recherche`, `doc6` trois occurrences de `note`, et chaque comparaison porte bien sur les mêmes données. L’empreinte calculée dans la sortie aide à repérer toute modification accidentelle du corpus.

### Fréquence documentaire

La fréquence documentaire `df(t)` compte les documents qui contiennent au moins une fois le terme `t`. Elle ne compte pas ses répétitions à l’intérieur d’un document.

In [3]:
var documentFrequency = vocabulary.ToDictionary(
    term => term,
    term => corpus.Count(pair => pair.Value.Contains(term, StringComparer.Ordinal)),
    StringComparer.Ordinal);
Console.WriteLine("Fréquences documentaires (df) :");
foreach (string term in vocabulary)
    Console.WriteLine($"  {term,-14} df={documentFrequency[term]}");
if (documentFrequency.Count != 21 || documentFrequency["note"] != 8 || documentFrequency["recherche"] != 3)
    throw new Exception("Les fréquences documentaires du corpus attendu ont changé.");
Console.WriteLine("Contrôles structurels du vocabulaire et des termes repères réussis.");

Fréquences documentaires (df) :


  classement     df=1


  code           df=1


  compression    df=2


  documents      df=1


  entropie       df=2


... (13 outputs left out)

  recherche      df=3


  shannon        df=1


  strategie      df=1


Contrôles structurels du vocabulaire et des termes repères réussis.


Lecture : `note` a df=8, donc il n’aide pas à distinguer un sujet ici ; `recherche` a df=3 et apparaît exactement dans les trois documents pertinents de la première requête. Les df de 2 mélangent des paires thématiques (`compression`, `langue`) et le terme `long`, qui traverse deux thèmes : la fréquence seule ne révèle pas le sens.

## 2. De la fréquence brute à l’IDF

Le comptage brut additionne les fréquences des termes demandés. TF-IDF multiplie la fréquence du terme dans le document par un poids inversement lié au nombre de documents qui le contiennent :

$$\operatorname{idf}(t)=\ln\!\left(\frac{N}{df(t)}\right).$$

Le carnet utilise explicitement `ln(N/df)`. Un terme présent partout reçoit donc un poids nul. Pour un terme absent du vocabulaire, notre contrat de requête est de l’ignorer : `Idf` renvoie zéro, et les requêtes vides ou inconnues classent tous les documents à égalité.

In [4]:
static double Idf(string term, int totalDocuments, IReadOnlyDictionary<string, int> df)
{
    if (!df.TryGetValue(term, out int frequency))
        return 0.0; // Terme hors vocabulaire : ignoré dans cette variante pédagogique.
    return Math.Log((double)totalDocuments / frequency);
}

Console.WriteLine($"idf(note) = {Idf("note", documentCount, documentFrequency):F12} (terme omniprésent)");
Console.WriteLine($"idf(recherche) = {Idf("recherche", documentCount, documentFrequency):F12} (df=3)");
Console.WriteLine($"idf(pertinence) = {Idf("pertinence", documentCount, documentFrequency):F12} (df=1)");
Console.WriteLine($"Politique OOV : idf(terme absent) = {Idf("terme-inconnu", documentCount, documentFrequency):F1}.");

idf(note) = 0.000000000000 (terme omniprésent)


idf(recherche) = 0.980829253012 (df=3)


idf(pertinence) = 2.079441541680 (df=1)


Politique OOV : idf(terme absent) = 0.0.


Comme l’illustre la sortie, l’IDF décroît quand `df` augmente : le terme omniprésent s’annule, le terme de df=3 garde un poids positif et un terme unique reçoit `ln(8)`. Pour toute requête, les termes absents sont ignorés plutôt que de provoquer une recherche impossible.

### Exercice 1 — recalculer un IDF

Implémentez `MonIdf` avec la formule précédente, `documentCount` et `documentFrequency`. Vérifiez le cas omniprésent et un terme rare. Pour un terme inconnu, conservez le contrat annoncé : poids nul.

In [5]:
// Exercice 1 — complétez la fonction, puis laissez les contrôles valider votre réponse.
static double? MonIdf(string term)
{
    // TODO étudiant : calculer ln(N / df(term)); retourner null tant que la réponse manque.
    return null;
}

double? idfNoteEtudiant = MonIdf("note");
double? idfRareEtudiant = MonIdf("pertinence");
double? idfOovEtudiant = MonIdf("terme-inconnu");
if (idfNoteEtudiant.HasValue && idfRareEtudiant.HasValue && idfOovEtudiant.HasValue)
{
    if (!double.IsFinite(idfNoteEtudiant.Value) || !double.IsFinite(idfRareEtudiant.Value) || !double.IsFinite(idfOovEtudiant.Value) ||
        Math.Abs(idfNoteEtudiant.Value) > 1e-12 ||
        Math.Abs(idfRareEtudiant.Value - Math.Log(documentCount)) > 1e-12 ||
        Math.Abs(idfOovEtudiant.Value) > 1e-12)
        throw new Exception("Vérifiez les valeurs exactes des cas omniprésent, rare et hors vocabulaire.");
    Console.WriteLine("Contrôle Exercice 1 réussi : note=0, rare=ln(8), OOV=0; toutes les réponses sont finies.");
}
else
{
    Console.WriteLine("Exercice 1 à compléter : le contrôle s'exécutera après votre réponse.");
}

Exercice 1 à compléter : le contrôle s'exécutera après votre réponse.


## 3. Trois classeurs, une règle d’ordre commune

Le premier score ne voit que les fréquences brutes. Le tri est déterministe : score décroissant, puis identifiant de document en ordre ordinal croissant. Cette règle rend explicites les ex æquo à zéro des requêtes sans correspondance.

In [6]:
static int TermFrequency(IReadOnlyList<string> tokens, string term) =>
    tokens.Count(token => StringComparer.Ordinal.Equals(token, term));

static double RawTfScore(IReadOnlyList<string> tokens, IEnumerable<string> query) =>
    query.Sum(term => TermFrequency(tokens, term));

static List<string> RankBy(IReadOnlyDictionary<string, double> scores) =>
    scores.OrderByDescending(pair => pair.Value)
        .ThenBy(pair => pair.Key, StringComparer.Ordinal)
        .Select(pair => pair.Key).ToList();

var queries = new (string Name, string[] Terms, HashSet<string> Relevant)[]
{
    ("Q1 recherche+note", new[] { "recherche", "note" }, new HashSet<string>(new[] { "doc1", "doc2", "doc3" }, StringComparer.Ordinal)),
    ("Q2 compression", new[] { "compression", "entropie" }, new HashSet<string>(new[] { "doc4", "doc5" }, StringComparer.Ordinal)),
    ("Q3 modeles", new[] { "modeles", "probabilistes" }, new HashSet<string>(new[] { "doc7", "doc8" }, StringComparer.Ordinal))
};

var rawQ1Scores = corpus.ToDictionary(pair => pair.Key,
    pair => RawTfScore(pair.Value, queries[0].Terms), StringComparer.Ordinal);
Console.WriteLine("Exemple 1 — classement TF brut pour Q1 (recherche + note) :");
Console.WriteLine(string.Join(" → ", RankBy(rawQ1Scores).Take(4)
    .Select(document => $"{document} ({rawQ1Scores[document]:F0})")));
if (rawQ1Scores["doc2"] != 3.0 || rawQ1Scores["doc6"] != 3.0 || rawQ1Scores["doc1"] != 2.0)
    throw new Exception("Les scores TF bruts repères ne correspondent plus au corpus source.");
Console.WriteLine("Fixture TF brut vérifiée : doc2=3, doc6=3, doc1=2.");

Exemple 1 — classement TF brut pour Q1 (recherche + note) :


doc2 (3) → doc6 (3) → doc1 (2) → doc3 (2)


Fixture TF brut vérifiée : doc2=3, doc6=3, doc1=2.


Le comptage brut place `doc6` haut parce qu’il contient trois occurrences de `note`, même s’il ne contient pas `recherche`. Une fréquence répétée d’un terme commun gonfle le score sans ajouter d’indication sur le thème.

### TF-IDF cosinus

Le vecteur documentaire utilise `tf × idf` sur le vocabulaire entier ; le vecteur de requête utilise la présence binaire des termes connus, pondérée par le même IDF. Le cosinus compare les directions des vecteurs et retourne zéro lorsque la requête n’a aucune coordonnée non nulle.

In [7]:
static Dictionary<string, double> TfIdfVector(
    IReadOnlyList<string> tokens, IReadOnlyList<string> terms, int totalDocuments,
    IReadOnlyDictionary<string, int> df)
{
    return terms.ToDictionary(term => term,
        term => TermFrequency(tokens, term) * Idf(term, totalDocuments, df), StringComparer.Ordinal);
}

static double TfIdfCosine(
    IReadOnlyList<string> query, IReadOnlyList<string> document,
    IReadOnlyList<string> terms, int totalDocuments, IReadOnlyDictionary<string, int> df)
{
    var queryWeights = query.Distinct(StringComparer.Ordinal)
        .Where(df.ContainsKey)
        .ToDictionary(term => term, term => Idf(term, totalDocuments, df), StringComparer.Ordinal);
    var documentWeights = TfIdfVector(document, terms, totalDocuments, df);
    double dot = queryWeights.Sum(pair => pair.Value * documentWeights[pair.Key]);
    double queryNorm = Math.Sqrt(queryWeights.Values.Sum(weight => weight * weight));
    double documentNorm = Math.Sqrt(documentWeights.Values.Sum(weight => weight * weight));
    return queryNorm > 0.0 && documentNorm > 0.0 ? dot / (queryNorm * documentNorm) : 0.0;
}

var tfidfQ1Scores = corpus.ToDictionary(pair => pair.Key,
    pair => TfIdfCosine(queries[0].Terms, pair.Value, vocabulary, documentCount, documentFrequency),
    StringComparer.Ordinal);
Console.WriteLine("Exemple 2 — classement TF-IDF cosinus pour Q1 :");
foreach (string document in RankBy(tfidfQ1Scores).Take(4))
    Console.WriteLine($"  {document} : {tfidfQ1Scores[document]:F6}");
if (Math.Abs(tfidfQ1Scores["doc3"] - 1.0) > 1e-12 ||
    Math.Abs(tfidfQ1Scores["doc2"] - 0.5549238723607618) > 1e-10 ||
    Math.Abs(tfidfQ1Scores["doc1"] - 0.2627553065654378) > 1e-10)
    throw new Exception("Les scores TF-IDF repères ne correspondent plus à la formule source.");
Console.WriteLine("Fixtures TF-IDF vérifiées : doc3=1, doc2≈0.554924, doc1≈0.262755.");

Exemple 2 — classement TF-IDF cosinus pour Q1 :


  doc3 : 1.000000


  doc2 : 0.554924


  doc1 : 0.262755


  doc4 : 0.000000


Fixtures TF-IDF vérifiées : doc3=1, doc2≈0.554924, doc1≈0.262755.


Le terme `note` n’a aucun poids dans l’un ou l’autre vecteur. Le classement place donc les documents qui contiennent `recherche` devant le document de jeux : le cosinus TF-IDF ne reprend pas le score artificiel de `doc6`.

## 4. BM25 : fréquence saturée et longueur normalisée

Nous implémentons la variante de ce cours avec `idf = ln(N/df)`, la même que dans le jumeau Python :

$$\operatorname{score}(d,q)=\sum_{t\in q}\operatorname{idf}(t)\;\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/\overline{dl})}.$$

`k1` rend le gain d’occurrences sous-linéaire ; `b` normalise la longueur du document. Cette variante donne exactement zéro au terme omniprésent. L’IDF `ln(N/df)` est distinct de l’IDF RSJ utilisée par `Lucene.Net.BM25Similarity`, `ln(1+(N-df+0,5)/(df+0,5))` : ce carnet montre les calculs de la première formule, il ne prétend pas reproduire le score d’un moteur de production.

In [8]:
const double K1 = 1.5;
const double B = 0.75;
double averageDocumentLength = lengths.Values.Average();

static double Bm25TermContribution(
    string term, IReadOnlyList<string> document, int totalDocuments,
    IReadOnlyDictionary<string, int> df, double averageLength, double k1, double b)
{
    int frequency = TermFrequency(document, term);
    if (frequency == 0)
        return 0.0;
    double lengthFactor = 1.0 - b + b * document.Count / averageLength;
    return Idf(term, totalDocuments, df) *
        (frequency * (k1 + 1.0)) / (frequency + k1 * lengthFactor);
}

static double Bm25Score(
    IEnumerable<string> query, IReadOnlyList<string> document, int totalDocuments,
    IReadOnlyDictionary<string, int> df, double averageLength, double k1, double b)
{
    return query.Sum(term => Bm25TermContribution(term, document, totalDocuments, df,
        averageLength, k1, b));
}

Console.WriteLine($"BM25 pédagogique : avgdl={averageDocumentLength:F3}, k1={K1:F2}, b={B:F2}.");
Console.WriteLine($"Contribution recherche dans doc1 : {Bm25TermContribution("recherche", corpus["doc1"], documentCount, documentFrequency, averageDocumentLength, K1, B):F12}.");
Console.WriteLine($"Contribution recherche dans doc2 : {Bm25TermContribution("recherche", corpus["doc2"], documentCount, documentFrequency, averageDocumentLength, K1, B):F12}.");
Console.WriteLine($"Contribution recherche dans doc3 : {Bm25TermContribution("recherche", corpus["doc3"], documentCount, documentFrequency, averageDocumentLength, K1, B):F12}.");
Console.WriteLine($"Contribution note dans doc6 : {Bm25TermContribution("note", corpus["doc6"], documentCount, documentFrequency, averageDocumentLength, K1, B):F3}.");
if (Math.Abs(averageDocumentLength - 5.25) > 1e-12 ||
    Math.Abs(Bm25TermContribution("recherche", corpus["doc1"], documentCount, documentFrequency, averageDocumentLength, K1, B) - 1.0023072658513992) > 1e-10 ||
    Math.Abs(Bm25TermContribution("recherche", corpus["doc2"], documentCount, documentFrequency, averageDocumentLength, K1, B) - 1.4229647193952506) > 1e-10 ||
    Math.Abs(Bm25TermContribution("recherche", corpus["doc3"], documentCount, documentFrequency, averageDocumentLength, K1, B) - 1.359565301204373) > 1e-10)
    throw new Exception("Les fixtures numériques BM25 repères ne correspondent plus au jumeau Python.");
Console.WriteLine("Fixtures BM25 vérifiées : avgdl=5.25 et contributions recherche(doc1/doc2/doc3) ≈ 1.002307/1.422965/1.359565.");

BM25 pédagogique : avgdl=5.250, k1=1.50, b=0.75.


Contribution recherche dans doc1 : 1.002307265851.


Contribution recherche dans doc2 : 1.422964719395.


Contribution recherche dans doc3 : 1.359565301204.


Contribution note dans doc6 : 0.000.


Fixtures BM25 vérifiées : avgdl=5.25 et contributions recherche(doc1/doc2/doc3) ≈ 1.002307/1.422965/1.359565.


La sortie matérialise le compromis : `doc2` bénéficie de deux occurrences de `recherche`, tandis que la longueur réduite de `doc3` amplifie sa contribution. `note` reste nul avant saturation et normalisation, parce que son IDF est nul.

### Exercice 2 — contribution d’un terme BM25

Complétez `MonBm25Term` pour un terme et un document. Le contrôle exige une contribution nulle pour `note` et une contribution positive pour `recherche` dans `doc2`. Gardez `null` tant que le calcul n’est pas écrit.

In [9]:
// Exercice 2 — complétez la formule d'un terme de BM25.
static double? MonBm25Term(string term, string documentId)
{
    // TODO étudiant : calculer f, le facteur de longueur, puis la contribution idf × saturation.
    return null;
}

double? noteEtudiant = MonBm25Term("note", "doc6");
double? rechercheEtudiant = MonBm25Term("recherche", "doc2");
double? rechercheDoc3Etudiant = MonBm25Term("recherche", "doc3");
if (noteEtudiant.HasValue && rechercheEtudiant.HasValue && rechercheDoc3Etudiant.HasValue)
{
    if (!double.IsFinite(noteEtudiant.Value) || !double.IsFinite(rechercheEtudiant.Value) || !double.IsFinite(rechercheDoc3Etudiant.Value) ||
        Math.Abs(noteEtudiant.Value) > 1e-12 ||
        Math.Abs(rechercheEtudiant.Value - 1.4229647193952506) > 1e-10 ||
        Math.Abs(rechercheDoc3Etudiant.Value - 1.359565301204373) > 1e-10)
        throw new Exception("Vérifiez les contributions BM25 numériques de note, doc2 et doc3.");
    Console.WriteLine("Contrôle Exercice 2 réussi : note=0; recherche(doc2)≈1.422965; recherche(doc3)≈1.359565; réponses finies.");
}
else
{
    Console.WriteLine("Exercice 2 à compléter : le contrôle s'exécutera après votre réponse.");
}

Exercice 2 à compléter : le contrôle s'exécutera après votre réponse.


## 5. Classer et mesurer le rappel

Les pertinences sont des ensembles de documents, pas un ordre de préférence. Nous conservons les trois requêtes, les documents pertinents de la source et `Recall@3` : nombre de pertinents dans les trois premiers, divisé par le nombre total de pertinents.

In [10]:
static List<string> TopK(IReadOnlyDictionary<string, double> scores, int k = 3) =>
    RankBy(scores).Take(Math.Max(0, k)).ToList();

static double RecallAtK(IReadOnlyList<string> ranked, ISet<string> relevant, int k)
{
    if (k <= 0 || relevant.Count == 0)
        return 0.0;
    return ranked.Take(k).Distinct(StringComparer.Ordinal).Count(relevant.Contains) / (double)relevant.Count;
}

static double PrecisionAtK(IReadOnlyList<string> ranked, ISet<string> relevant, int k)
{
    if (k <= 0)
        return 0.0;
    return ranked.Take(k).Distinct(StringComparer.Ordinal).Count(relevant.Contains) / (double)k;
}

static double MeanReciprocalRank(IReadOnlyList<string> ranked, ISet<string> relevant)
{
    var uniqueRanking = ranked.Distinct(StringComparer.Ordinal).ToArray();
    for (int index = 0; index < uniqueRanking.Length; index++)
        if (relevant.Contains(uniqueRanking[index]))
            return 1.0 / (index + 1);
    return 0.0;
}

static double BinaryNdcgAtK(IReadOnlyList<string> ranked, ISet<string> relevant, int k)
{
    if (k <= 0 || relevant.Count == 0)
        return 0.0;
    var uniqueRanking = ranked.Distinct(StringComparer.Ordinal).ToArray();
    double dcg = uniqueRanking.Take(k).Select((document, index) =>
        relevant.Contains(document) ? 1.0 / Math.Log2(index + 2) : 0.0).Sum();
    int idealRelevant = Math.Min(k, relevant.Count);
    double idealDcg = Enumerable.Range(0, idealRelevant)
        .Sum(index => 1.0 / Math.Log2(index + 2));
    return idealDcg > 0.0 ? dcg / idealDcg : 0.0;
}

Console.WriteLine("Métriques définies : Recall@k, Precision@k, MRR et nDCG binaire@k; doublons dédupliqués, k<=0/jeu pertinent vide donnent 0.");

Métriques définies : Recall@k, Precision@k, MRR et nDCG binaire@k; doublons dédupliqués, k<=0/jeu pertinent vide donnent 0.


Les métriques sont définies avant leur tableau d’application. `Precision@k` divise par la valeur fixée de `k`, `Recall@k` par la taille du jeu pertinent ; `MRR` mesure la première occurrence pertinente et `nDCG` binaire compare l’ordre à un ordre idéal. Les jeux pertinents vides donnent zéro par convention explicite. Les identifiants répétés sont dédupliqués : Recall et Precision comptent les identifiants distincts du préfixe ; MRR et nDCG parcourent une liste de résultats uniques.

### Exemple 3 — classements et tableau de Recall@3

Nous comparons les trois classeurs sur les mêmes requêtes. Les ex æquo sont ordonnés par identifiant de document ; un document non pertinent à score nul peut apparaître au bord du top-3, ce qui n’en fait pas une recommandation.

In [11]:
Dictionary<string, double> ScoresFor(string method, string[] query) => corpus.ToDictionary(
    pair => pair.Key,
    pair => method switch
    {
        "TF brut" => RawTfScore(pair.Value, query),
        "TF-IDF" => TfIdfCosine(query, pair.Value, vocabulary, documentCount, documentFrequency),
        "BM25" => Bm25Score(query, pair.Value, documentCount, documentFrequency,
            averageDocumentLength, K1, B),
        _ => throw new ArgumentOutOfRangeException(nameof(method))
    }, StringComparer.Ordinal);

string[] rankerNames = { "TF brut", "TF-IDF", "BM25" };
var rankedByQueryAndMethod = new Dictionary<(string Query, string Method), List<string>>();
foreach (var query in queries)
{
    Console.WriteLine($"{query.Name} — pertinents : {string.Join(", ", query.Relevant.OrderBy(id => id, StringComparer.Ordinal))}");
    foreach (string method in rankerNames)
    {
        var ranked = TopK(ScoresFor(method, query.Terms));
        rankedByQueryAndMethod[(query.Name, method)] = ranked;
        Console.WriteLine($"  {method,-7} : " + string.Join("  ", ranked.Select(document =>
            document + (query.Relevant.Contains(document) ? "*" : ""))));
    }
}

Console.WriteLine("\nTableau Recall@3 :");
Console.WriteLine($"{"Classeurs",-12} {queries[0].Name,18} {queries[1].Name,18} {queries[2].Name,18} {"Moyenne",10}");
foreach (string method in rankerNames)
{
    double[] values = queries.Select(query => RecallAtK(
        rankedByQueryAndMethod[(query.Name, method)], query.Relevant, 3)).ToArray();
    Console.WriteLine($"{method,-12} {values[0],18:F3} {values[1],18:F3} {values[2],18:F3} {values.Average(),10:F3}");
}

var rawQ1Top = rankedByQueryAndMethod[(queries[0].Name, "TF brut")];
var tfidfQ1Top = rankedByQueryAndMethod[(queries[0].Name, "TF-IDF")];
var bm25Q1Top = rankedByQueryAndMethod[(queries[0].Name, "BM25")];
if (!rawQ1Top.SequenceEqual(new[] { "doc2", "doc6", "doc1" }) ||
    !tfidfQ1Top.SequenceEqual(new[] { "doc3", "doc2", "doc1" }) ||
    !bm25Q1Top.SequenceEqual(new[] { "doc2", "doc3", "doc1" }))
    throw new Exception("Les classements Q1 ne correspondent plus aux fixtures du carnet source.");
Console.WriteLine("Fixtures numériques Q1 validées : TF brut doc2/doc6/doc1, TF-IDF doc3/doc2/doc1, BM25 doc2/doc3/doc1.");

Q1 recherche+note — pertinents : doc1, doc2, doc3


  TF brut : doc2*  doc6  doc1*


  TF-IDF  : doc3*  doc2*  doc1*


  BM25    : doc2*  doc3*  doc1*


Q2 compression — pertinents : doc4, doc5


  TF brut : doc5*  doc4*  doc1


... (8 outputs left out)

TF brut                   0.667              1.000              1.000      0.889


TF-IDF                    1.000              1.000              1.000      1.000


BM25                      1.000              1.000              1.000      1.000


Fixtures numériques Q1 validées : TF brut doc2/doc6/doc1, TF-IDF doc3/doc2/doc1, BM25 doc2/doc3/doc1.


Sur Q1, le TF brut laisse l’intrus `doc6` prendre la place de `doc3`, car `note` y est répété. TF-IDF et BM25 excluent l’intrus : leur IDF annule `note`. Q2 et Q3 retrouvent leurs deux documents pertinents pour les trois méthodes ; les autres documents à score zéro restent départagés par identifiant.

### Exercice 3 — vérifier Recall@k

Écrivez `MonRecallAtK` et contrôlez le classement brut de Q1. À `k=3`, il doit retrouver deux des trois documents pertinents. Cette mesure ne doit pas être confondue avec précision ou ordre de préférence.

In [12]:
// Exercice 3 — complétez le calcul du rappel à k.
static double? MonRecallAtK(IReadOnlyList<string> ranked, ISet<string> relevant, int k)
{
    // TODO étudiant : pertinents du préfixe / total des pertinents; null indique un stub non résolu.
    return null;
}

double? recallEtudiant = MonRecallAtK(rawQ1Top, queries[0].Relevant, 3);
double? recallDoublonsEtudiant = MonRecallAtK(new[] { "doc2", "doc2", "doc6" }, queries[0].Relevant, 3);
double? recallKZeroEtudiant = MonRecallAtK(rawQ1Top, queries[0].Relevant, 0);
if (recallEtudiant.HasValue && recallDoublonsEtudiant.HasValue && recallKZeroEtudiant.HasValue)
{
    if (!double.IsFinite(recallEtudiant.Value) || !double.IsFinite(recallDoublonsEtudiant.Value) || !double.IsFinite(recallKZeroEtudiant.Value) ||
        Math.Abs(recallEtudiant.Value - 2.0 / 3.0) > 1e-12 ||
        Math.Abs(recallDoublonsEtudiant.Value - 1.0 / 3.0) > 1e-12 ||
        Math.Abs(recallKZeroEtudiant.Value) > 1e-12)
        throw new Exception("Vérifiez Recall@3, la déduplication du préfixe et le cas k=0.");
    Console.WriteLine("Contrôle Exercice 3 réussi : Q1=2/3; un doublon ne compte qu'une fois (1/3); k=0 donne 0; réponses finies.");
}
else
{
    Console.WriteLine("Exercice 3 à compléter : le contrôle s'exécutera après votre réponse.");
}

Exercice 3 à compléter : le contrôle s'exécutera après votre réponse.


Le rappel décrit une proportion retrouvée, pas une préférence entre deux résultats tous deux pertinents. La moyenne des trois requêtes est utile au petit banc d’essai, mais elle ne transforme pas huit documents et trois requêtes en benchmark général.

### Exemple 4 — métriques binaires et permutation de pertinents

À `k` fixé, une permutation des mêmes `k` documents ne change pas `Precision@k` : cette métrique ne dépend que du nombre de pertinents dans le préfixe. Avec des jugements binaires, MRR et nDCG ne distinguent pas non plus deux documents qui ont tous deux la même pertinence. Le petit contre-exemple suivant contrôle ces invariances.

In [13]:
var relevantPair = new HashSet<string>(new[] { "A", "B" }, StringComparer.Ordinal);
string[] orderABX = { "A", "B", "X" };
string[] orderBAX = { "B", "A", "X" };
double[] metricsABX =
{
    PrecisionAtK(orderABX, relevantPair, 3),
    RecallAtK(orderABX, relevantPair, 3),
    MeanReciprocalRank(orderABX, relevantPair),
    BinaryNdcgAtK(orderABX, relevantPair, 3)
};
double[] metricsBAX =
{
    PrecisionAtK(orderBAX, relevantPair, 3),
    RecallAtK(orderBAX, relevantPair, 3),
    MeanReciprocalRank(orderBAX, relevantPair),
    BinaryNdcgAtK(orderBAX, relevantPair, 3)
};
if (!metricsABX.Zip(metricsBAX).All(pair => Math.Abs(pair.First - pair.Second) < 1e-12) ||
    Math.Abs(metricsABX[0] - 2.0 / 3.0) > 1e-12 || metricsABX[2] != 1.0 || metricsABX[3] != 1.0)
    throw new Exception("La permutation de deux pertinents binaires doit préserver ces métriques.");
Console.WriteLine("A,B,X et B,A,X, pertinents {A,B} :");
Console.WriteLine($"  Precision@3={metricsABX[0]:F3}; Recall@3={metricsABX[1]:F3}; MRR={metricsABX[2]:F3}; nDCG@3={metricsABX[3]:F3}.");
Console.WriteLine("Les quatre métriques restent identiques après permutation de A et B.");
string[] orderXAB = { "X", "A", "B" };
double pXAB = PrecisionAtK(orderXAB, relevantPair, 3);
double rXAB = RecallAtK(orderXAB, relevantPair, 3);
double mrrXAB = MeanReciprocalRank(orderXAB, relevantPair);
double ndcgXAB = BinaryNdcgAtK(orderXAB, relevantPair, 3);
if (pXAB != metricsABX[0] || rXAB != metricsABX[1] || mrrXAB != 0.5 || !(ndcgXAB < metricsABX[3]))
    throw new Exception("Precision/Recall doivent garder le compte tandis que MRR/nDCG voient X devant les pertinents.");
Console.WriteLine($"X,A,B (pertinents {{A,B}}) : Precision@3={pXAB:F3}, Recall@3={rXAB:F3}, MRR={mrrXAB:F3}, nDCG@3={ndcgXAB:F3}.");
var singleRelevant = new HashSet<string>(new[] { "A" }, StringComparer.Ordinal);
if (BinaryNdcgAtK(new[] { "A", "A" }, singleRelevant, 2) != 1.0)
    throw new Exception("Un identifiant dupliqué ne doit pas compter deux fois dans nDCG.");
Console.WriteLine("Contrôle doublon nDCG : [A,A] avec pertinent {A} reste normalisé à 1.");

A,B,X et B,A,X, pertinents {A,B} :


  Precision@3=0.667; Recall@3=1.000; MRR=1.000; nDCG@3=1.000.


Les quatre métriques restent identiques après permutation de A et B.


X,A,B (pertinents {A,B}) : Precision@3=0.667, Recall@3=1.000, MRR=0.500, nDCG@3=0.693.


Contrôle doublon nDCG : [A,A] avec pertinent {A} reste normalisé à 1.


Le test confirme les limites précises des métriques : la précision à `k` fixe compte les pertinents du préfixe sans regarder leur ordre ; Recall compte le jeu retrouvé ; MRR et nDCG binaires voient l’ordre pertinent/non pertinent, pas la permutation entre deux résultats de même grade. Pour exprimer une préférence entre `A` et `B`, il faut des jugements gradués ou une utilité externe qui les différencie.

## 6. IDF de cette leçon et variante RSJ

Une formule courante de BM25 dans des moteurs tels que Lucene emploie l’IDF RSJ `ln(1+(N-df+0,5)/(df+0,5))`. Elle ne donne pas zéro au terme omniprésent, alors que la variante `ln(N/df)` utilisée ici le fait. Le contrôle numérique isole la différence, sans installer ni appeler Lucene.

In [14]:
double luceneRsjForUbiquitousTerm = Math.Log(1.0 +
    (documentCount - documentFrequency["note"] + 0.5) /
    (documentFrequency["note"] + 0.5));
if (Math.Abs(Idf("note", documentCount, documentFrequency)) > 1e-12 ||
    Math.Abs(luceneRsjForUbiquitousTerm - 0.05715841383994863) > 1e-12)
    throw new Exception("Les deux conventions d'IDF doivent rester distinctes.");
Console.WriteLine($"IDF de ce carnet pour note (N=df=8) : {Idf("note", documentCount, documentFrequency):F12}.");
Console.WriteLine($"IDF RSJ/Lucene pour N=df=8 : {luceneRsjForUbiquitousTerm:F12}.");
Console.WriteLine("Moteur Lucene non appelé : comparaison de formule seulement.");

IDF de ce carnet pour note (N=df=8) : 0.000000000000.


IDF RSJ/Lucene pour N=df=8 : 0.057158413840.


Moteur Lucene non appelé : comparaison de formule seulement.


Cette différence est intentionnelle. La formule du jumeau Python est utile pour isoler l’annulation du terme omniprésent ; le score BM25 de Lucene utilise l’ajustement RSJ ci-dessus. Les deux variantes ne sont pas interchangeables et leurs rangs peuvent différer. Aucun moteur de recherche industriel n’est une dépendance cachée de ce carnet.

## 7. Cas limites : termes absents, requête vide et ex æquo

La politique hors vocabulaire est testée à la frontière de l’algorithme : une requête vide, inconnue ou composée d’un terme connu et d’un terme absent doit être définie. Tous les scores nuls retombent sur le tri déterministe par identifiant.

In [15]:
var emptyQueryScores = ScoresFor("TF-IDF", Array.Empty<string>());
var unknownQueryScores = ScoresFor("BM25", new[] { "terme-inconnu" });
var mixedQueryScores = ScoresFor("TF-IDF", new[] { "recherche", "terme-inconnu" });
var knownOnlyScores = ScoresFor("TF-IDF", new[] { "recherche" });
var duplicatedRank = new[] { "doc2", "doc2", "doc6" };
double duplicateRecall = RecallAtK(duplicatedRank, queries[0].Relevant, 3);
string[] expectedTieOrder = corpus.Keys.OrderBy(id => id, StringComparer.Ordinal).ToArray();
if (emptyQueryScores.Values.Any(score => score != 0.0) ||
    unknownQueryScores.Values.Any(score => score != 0.0) ||
    !RankBy(emptyQueryScores).SequenceEqual(expectedTieOrder) ||
    !RankBy(unknownQueryScores).SequenceEqual(expectedTieOrder) ||
    mixedQueryScores.Any(pair => Math.Abs(pair.Value - knownOnlyScores[pair.Key]) > 1e-12) ||
    Math.Abs(duplicateRecall - 1.0 / 3.0) > 1e-12 ||
    RecallAtK(expectedTieOrder, new HashSet<string>(), 3) != 0.0 ||
    PrecisionAtK(expectedTieOrder, new HashSet<string>(), 3) != 0.0)
    throw new Exception("Un cas vide/OOV ou la règle des ex æquo a changé.");
Console.WriteLine("Requête vide : 8 scores nuls, ordre doc1→doc8.");
Console.WriteLine("Requête OOV : 8 scores nuls, ordre doc1→doc8; terme OOV ignoré dans la requête mixte.");
Console.WriteLine("Jeu pertinent vide : Recall@3=0 et Precision@3=0 par convention explicite.");
Console.WriteLine($"Préfixe avec doublon [doc2, doc2, doc6] : Recall@3={duplicateRecall:F3}; chaque identifiant compte au plus une fois.");

Requête vide : 8 scores nuls, ordre doc1→doc8.


Requête OOV : 8 scores nuls, ordre doc1→doc8; terme OOV ignoré dans la requête mixte.


Jeu pertinent vide : Recall@3=0 et Precision@3=0 par convention explicite.


Préfixe avec doublon [doc2, doc2, doc6] : Recall@3=0.333; chaque identifiant compte au plus une fois.


Le contrat est ainsi observable : ignorer un terme absent ne provoque pas d’exception ; la requête vide ou entièrement inconnue produit une égalité de scores ; le tri ordinal rend cette égalité reproductible. Un jeu pertinent vide donne zéro au lieu d’une division par zéro.

## 8. Figure : rappel moyen sur ce petit jeu fixe

In [16]:
static string EscapeXml(string value) =>
    System.Security.SecurityElement.Escape(value) ?? string.Empty;

static string BuildRecallSvg(IReadOnlyList<string> names, IReadOnlyList<double> values)
{
    const int width = 780;
    const int height = 430;
    const double left = 86, top = 66, plotWidth = 640, plotHeight = 270;
    const double maximum = 1.15;
    var svg = new StringBuilder();
    svg.AppendLine($"<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{width}\" height=\"{height}\" viewBox=\"0 0 {width} {height}\">");
    svg.AppendLine("<rect x=\"0\" y=\"0\" width=\"100%\" height=\"100%\" fill=\"#ffffff\"/>");
    svg.AppendLine("<text x=\"390\" y=\"32\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"20\" fill=\"#17212b\">Recall@3 moyen — 8 documents, 3 requêtes</text>");
    for (int tick = 0; tick <= 5; tick++)
    {
        double value = tick * 0.2;
        double y = top + plotHeight * (1.0 - value / maximum);
        svg.AppendLine($"<line x1=\"{left}\" y1=\"{y:F1}\" x2=\"{left + plotWidth}\" y2=\"{y:F1}\" stroke=\"#d5dce3\" stroke-width=\"1\"/>");
        svg.AppendLine($"<text x=\"{left - 12}\" y=\"{y + 5:F1}\" text-anchor=\"end\" font-family=\"sans-serif\" font-size=\"13\" fill=\"#263238\">{value:F1}</text>");
    }
    svg.AppendLine($"<line x1=\"{left}\" y1=\"{top}\" x2=\"{left}\" y2=\"{top + plotHeight}\" stroke=\"#263238\"/>");
    svg.AppendLine($"<line x1=\"{left}\" y1=\"{top + plotHeight}\" x2=\"{left + plotWidth}\" y2=\"{top + plotHeight}\" stroke=\"#263238\"/>");
    svg.AppendLine($"<text transform=\"translate(25 {top + plotHeight / 2:F1}) rotate(-90)\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"14\" fill=\"#263238\">Recall@3 moyen</text>");
    string[] colors = { "#a83232", "#2563a6", "#217a46" };
    double slot = plotWidth / names.Count;
    double barWidth = 112;
    for (int index = 0; index < names.Count; index++)
    {
        double barHeight = values[index] / maximum * plotHeight;
        double x = left + slot * index + (slot - barWidth) / 2;
        double y = top + plotHeight - barHeight;
        svg.AppendLine($"<rect x=\"{x:F1}\" y=\"{y:F1}\" width=\"{barWidth}\" height=\"{barHeight:F1}\" fill=\"{colors[index]}\"/>");
        svg.AppendLine($"<text x=\"{x + barWidth / 2:F1}\" y=\"{y - 10:F1}\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"15\" font-weight=\"bold\" fill=\"#17212b\">{values[index]:F3}</text>");
        svg.AppendLine($"<text x=\"{x + barWidth / 2:F1}\" y=\"{top + plotHeight + 25:F1}\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"14\" fill=\"#17212b\">{EscapeXml(names[index])}</text>");
    }
    svg.AppendLine("<text x=\"390\" y=\"408\" text-anchor=\"middle\" font-family=\"sans-serif\" font-size=\"12\" fill=\"#46515c\">Moyenne arithmétique de Recall@3 sur les trois requêtes étiquetées.</text>");
    svg.AppendLine("</svg>");
    return svg.ToString();
}

var recallMeans = rankerNames.Select(method => queries.Average(query => RecallAtK(
    rankedByQueryAndMethod[(query.Name, method)], query.Relevant, 3))).ToArray();
string recallSvg = BuildRecallSvg(rankerNames, recallMeans);
display(new PlotSvg(recallSvg));
Console.WriteLine("Figure SVG calculée depuis les mêmes sorties Recall@3 : TF brut=8/9, TF-IDF=1, BM25=1.");

Recall@3 moyen — 8 documents, 3 requêtes 
 
 0.0 
 
 0.2 
 
 0.4 
 
 0.6 
 
 0.8 
 
 1.0 
 
 
 Recall@3 moyen 
 
 0.889 
 TF brut 
 
 1.000 
 TF-IDF 
 
 1.000 
 BM25 
 Moyenne arithmétique de Recall@3 sur les trois requêtes étiquetées.

Figure SVG calculée depuis les mêmes sorties Recall@3 : TF brut=8/9, TF-IDF=1, BM25=1.


Le graphique reprend les moyennes calculées par les métriques, pas des valeurs saisies séparément. Il résume uniquement les huit documents et les trois requêtes de ce corpus construit ; il ne mesure ni une satisfaction utilisateur ni la qualité générale d’un moteur.

## À retenir

- `df` compte les documents contenant un terme ; avec `ln(N/df)`, le terme `note` à df=N a un IDF nul.
- TF-IDF cosinus et BM25 utilisent la rareté, mais leurs scores et leurs premiers rangs diffèrent : Q1 place `doc3` devant avec TF-IDF et `doc2` avec BM25.
- Le BM25 du carnet est la variante pédagogique du jumeau Python. Son IDF n’est pas le RSJ de Lucene.
- Recall@k ne mesure que la fraction de pertinents retrouvés. Precision@k à k fixé compte les pertinents dans le préfixe ; MRR et nDCG binaires ne départagent pas deux documents pertinents de même grade.
- Un classement nul reste ordonné par identifiant, et les termes OOV sont ignorés selon un contrat explicite.

## Vérifiez votre compréhension

1. Pourquoi `idf(note)` est-il nul, et quel effet cela a-t-il sur `doc6` pour Q1 ?
2. Quelle différence de calcul distinguer entre l’IDF de ce carnet et l’IDF RSJ de Lucene ?
3. Pourquoi Precision@3 ne change-t-elle pas quand on échange deux documents du top-3 ?
4. Que savent distinguer MRR et nDCG binaires lorsque les deux documents pertinents sont permutés ?

<details><summary>Réponses</summary>

1. `note` apparaît dans les huit documents, donc `ln(8/8)=0`. Les trois occurrences de `note` dans `doc6` ne contribuent plus au score pondéré.
2. Ce carnet emploie `ln(N/df)` : pour `N=df=8`, le poids vaut zéro. L’IDF RSJ de Lucene ajoute les demi-comptes et un `1` sous le logarithme ; pour ce cas, il vaut `ln(18/17)`, donc il n’est pas nul.
3. La précision à k fixe est le nombre de pertinents dans les k premiers divisé par k ; permuter ces k documents ne change pas le compte.
4. Avec des gains binaires identiques, ils voient la position pertinente/non pertinente, mais pas l’identité de deux résultats pertinents de même grade. Pour les départager, il faut des jugements gradués ou une utilité qui les distingue.

</details>

## Pour aller plus loin

- Le [jumeau Python](06-Recherche-Information-TFIDF-BM25.ipynb) expose les mêmes corpus, paramètres et rangs avec les mêmes formules.
- La référence classique est *Introduction to Information Retrieval*, Manning, Raghavan et Schütze, chapitre 6. Pour une variante de production, lire la formule et la configuration réellement utilisées par le moteur choisi ; le nom BM25 seul ne garantit pas des scores identiques.
- Le [carnet 07 — vecteurs de mots](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb) passe du classement lexical aux représentations par cooccurrence.